# Neural PM flux model — co-energy residual on the analytic baseline

## Why this version replaces the old one

The previous version of this notebook trained a network to predict the flux
vector `flux_pm(omega, i)` directly, from scratch, with no connection to the
existing analytic `ConstantFlux` model. That caused two real problems,
diagnosed in a later investigation of this library's fault-tolerant
optimizer:

1. **Inductance wasn't guaranteed symmetric.** `NeuralFlux.inductance()`
   computed `L_stat + d(flux_pred)/d(i)` as the Jacobian of an arbitrary
   4-output network — nothing forces a Jacobian of an unconstrained vector
   field to be symmetric, but a real inductance matrix must be (Maxwell
   reciprocity).
2. **Flux had no physical anchor.** The network's raw output was used
   directly, with no fallback to the analytic model. Combined with (1),
   this meant the optimizer (`IndependentOptimizer`, hunting for whatever
   current maximizes the torque quadratic form `i^T A i`) could — and
   empirically did — find operating points 3-4 standard deviations outside
   the training data where the network's Jacobian took on large, physically
   meaningless values, producing "torque" figures (~13 Nm vs. a real ~7-8
   Nm ceiling) that were pure extrapolation artifacts. A derivative of a
   fitted function is far less trustworthy outside training data than the
   function's own values, and here that untrustworthy derivative was being
   actively maximized inside a quadratic objective — close to the
   worst-case setup for an extrapolation failure.

## The fix: a scalar co-energy residual, anchored to the analytic model

Instead of predicting the flux vector from scratch, the network predicts a
single **scalar magnetic co-energy residual** `W_res(omega, i)` on top of
the existing analytic `ConstantFlux` baseline (`flux_pm`, `L_stat`, both
identified by `utils.flux_fit.fit_pm_flux` via joint least squares against
the same measured data used here — see `ieee_machine2_params()`):

```
flux(omega, i)       = flux_pm + L_stat @ i + d(W_res)/di
inductance(omega, i)  = L_stat + d^2(W_res)/di^2
```

This gives two properties by construction, not by hoping training gets
there:

- **Inductance is exactly symmetric.** The Hessian of any scalar function
  is symmetric; `NeuralFlux.inductance()` now computes
  `W1.T @ diag(w2 * act''(z)) @ W1` directly from the network's own
  weights, which is symmetric for any trained weights whatsoever.
- **Flux/inductance are anchored, not free-floating.** With the network's
  own contribution near zero (as at initialization, or wherever training
  keeps it small), both fall back to the analytic model's own values —
  which are well-behaved for any current — rather than to an ungrounded
  raw MLP output. This mirrors the existing `NeuralTorqueResidual`'s
  additive-correction pattern (`neural_pmsm5phase`), which never showed
  the extrapolation failure above, precisely because it's a single additive
  correction on a sound physical baseline rather than a derivative of an
  unconstrained function feeding a quadratic objective.

Both the gradient and the Hessian formulas used here were verified against
`torch.autograd` (first and second derivatives) to float precision before
this notebook was written this way — see `NeuralFlux`'s docstring in
`current_setpoints/models/machines.py`.

## Data

Same 174-row measured dataset as before (`data/aggregated_file_means.csv`),
same train/val/test split, same joint voltage+torque physics loss
philosophy — only the network's output and what it's added to have
changed. Known limitation, unchanged by this retraining: the dataset's
`id3`/`iq3` coverage is only ±10A (a third of `curr_max=30`), so no amount
of retraining makes this model trustworthy at large third-harmonic current
— that's a real data limitation, not a fixable modeling choice. What this
retraining fixes is that the model now degrades toward the physically sound
baseline outside its training data, instead of toward whatever an
unconstrained Jacobian happens to produce there.

In [ ]:
import copy
import os
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_params, ieee_machine2, NeuralFlux, _build_cross_coupling
from current_setpoints.models.machines import PMSMDrive
from current_setpoints.utils import (
    NeuralFluxPredictor,
    load_aggregated_csv_data,
    load_neural_flux_model,
)

torch.manual_seed(42)
np.random.seed(42)

## Data and fixed physics operators

`R_stat`, `L_stat`, `flux_pm` come straight from `ieee_machine2_params()` (the
same joint-least-squares identification as `ConstantFlux` uses) and are held
fixed during training as the baseline the network corrects. `J` is the
harmonic cross-coupling matrix, `A = k*(J@L+L@J^T)` the same quadratic torque
matrix `PMSMDrive.torque()` uses.

In [ ]:
AGGREGATED_FILE_PATH = "../data/aggregated_file_means.csv"
COLUMN_MAP = {c: c for c in ["omega", "id1", "iq1", "id3", "iq3", "ud1", "uq1", "ud3", "uq3", "torq"]}
INPUT_SIZE = 5
OUTPUT_SIZE = 1  # scalar co-energy residual -- NOT the flux vector (see notebook overview)
TEST_SIZE = 0.15
RANDOM_STATE = 42
VAL_SIZE = 0.20

# The dataset is tiny (174 rows, full-batch training); CPU is faster here
# than GPU due to per-kernel-launch overhead dominating at this scale.
DEVICE = torch.device("cpu")

HIDDEN_SIZE = 24
ACTIVATION = "gelu"
LR = 5e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 15000
PATIENCE = 1000

# Voltage-vs-torque loss trade-off (both normalized by their own variance
# first, so this is a genuine relative weight, not a units fudge). Tried
# {1, 3, 6, 10}: v_weight=1 gives torque RMSE 0.24 Nm but voltage RMSE 0.98V
# (worse than the analytic baseline's 0.55V); v_weight=6 brings voltage back
# to near-baseline (0.62V) while torque RMSE (0.53 Nm) stays well below the
# baseline's 0.87 Nm. Voltage is a hard optimizer CONSTRAINT while torque is
# the objective, so erring toward voltage accuracy is the safer default here
# -- raise/lower this and retrain if that priority should shift.
VOLT_WEIGHT = 6.0

MODEL_SAVE_PATH = "../weights/FluxNN_Weights.pth"
SCALER_SAVE_PATH = "../weights/FluxNN_Scaler.npy"

params = ieee_machine2_params()
R_stat = torch.from_numpy(params.R_stat).float().to(DEVICE)
L_stat = torch.from_numpy(params.L_stat).float().to(DEVICE)
flux_pm = torch.from_numpy(params.flux_pm).float().to(DEVICE)
J = torch.from_numpy(_build_cross_coupling(2)).float().to(DEVICE)
k = 5 * params.n_ppairs / 4.0
JL = J @ L_stat
A = k * (J @ L_stat + L_stat @ J.T)

df = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)
X = df[["omega", "id1", "iq1", "id3", "iq3"]].to_numpy(dtype=np.float32)
volt = df[["ud1", "uq1", "ud3", "uq3"]].to_numpy(dtype=np.float32)
torq = df[["torq"]].to_numpy(dtype=np.float32)

X_trainval, X_test, volt_trainval, volt_test, torq_trainval, torq_test = train_test_split(
    X, volt, torq, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train, X_val, volt_train, volt_val, torq_train, torq_val = train_test_split(
    X_trainval, volt_trainval, torq_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")
print(f"Using compute device: {DEVICE}")

## Model, analytic gradient/Hessian, and physics loss

The network (`NeuralFluxPredictor`, `output_size=1`) predicts the scalar
co-energy residual `W_res(omega, i)`. Gradient and Hessian w.r.t. current are
computed analytically from the network's own weights (single hidden layer:
`fc1 -> GELU -> fc2`) rather than via `torch.autograd.grad`/double backprop —
this is the exact same closed-form used by `NeuralFlux` at inference
(verified to match `torch.autograd` to float precision beforehand), so
training and inference are guaranteed consistent by construction, and it's
simpler and faster than double backprop. GELU only (fixed choice, matching
the previous architecture search's winner — that search was about capacity/
activation for predicting the flux directly, not really affected by this
reparameterization).

In [ ]:
import math


def _gelu_deriv(z):
    return 0.5 * (1.0 + torch.erf(z / math.sqrt(2.0))) + z * torch.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)


def _gelu_second_deriv(z):
    return (2.0 - z * z) * torch.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)


def batched_grad_hessian(model, x_normed, scaler_scale):
    """Analytic per-sample gradient and Hessian of the network's scalar
    output w.r.t. current, matching NeuralFlux's numpy inference formula
    exactly (W1.T @ diag(w2*act''(z)) @ W1 for the Hessian -- symmetric by
    construction). Ordinary tensor ops, differentiable w.r.t. the weights
    like any other forward pass, so normal backprop trains through this
    correctly -- no autograd.grad/double backprop needed.

    Returns (grad_i, hess_i): d(W_res)/di, d^2(W_res)/di^2, shapes (N, dim)
    and (N, dim, dim) -- already restricted to current dims (omega dropped)
    and divided through by the scaler scale (chain rule for x_normed).
    """
    W1 = model.fc1.weight  # (hidden, 5)
    b1 = model.fc1.bias  # (hidden,)
    w2 = model.fc2.weight.reshape(-1)  # (hidden,) -- output_size == 1
    scale = torch.as_tensor(scaler_scale, dtype=torch.float32)

    z = x_normed @ W1.T + b1  # (N, hidden)
    d_act = _gelu_deriv(z)
    d2_act = _gelu_second_deriv(z)

    grad_normed = (w2 * d_act) @ W1  # (N, 5)
    grad_x = grad_normed / scale
    grad_i = grad_x[:, 1:]  # (N, dim)

    weighted = (w2 * d2_act).unsqueeze(-1) * W1  # (N, hidden, 5)
    hess_normed = torch.einsum("hp,nhq->npq", W1, weighted)  # (N, 5, 5), symmetric per sample
    hess_x = hess_normed / (scale.view(1, -1, 1) * scale.view(1, 1, -1))
    hess_i = hess_x[:, 1:, 1:]  # (N, dim, dim)
    return grad_i, hess_i


def physics_loss(model, x_normed, i, omega, volt_meas, torq_meas, var_v, var_t):
    """Voltage/torque loss with flux = flux_pm + L_stat@i + grad(W_res),
    inductance = L_stat + hess(W_res), matching NeuralFlux/PMSMDrive exactly.
    A_batch here MUST match PMSMDrive.torque()'s formula (J@L + L@J.T, NOT
    (J@L)^T -- only equal when L is symmetric) -- the earlier version of
    this notebook had exactly this bug, silently degrading real test
    performance behind an artificially good training-time self-evaluation.

    The combined loss applies VOLT_WEIGHT to the (variance-normalized)
    voltage term -- see its definition above for why (voltage is a hard
    optimizer constraint, torque merely the objective).
    """
    grad_i, hess_i = batched_grad_hessian(model, x_normed, SCALER_SCALE)
    flux_pred = flux_pm.unsqueeze(0) + i @ L_stat.T + grad_i  # (N, dim)
    L_batch = L_stat.unsqueeze(0) + hess_i  # (N, dim, dim)

    JL_batch = torch.einsum("jl,nlk->njk", J, L_batch)
    LJt_batch = torch.einsum("npq,rq->npr", L_batch, J)
    v_pred = (
        i @ R_stat.T
        + omega * torch.einsum("njk,nk->nj", JL_batch, i)
        + omega * torch.einsum("jl,nl->nj", J, flux_pred)
    )
    loss_v = nn.functional.mse_loss(v_pred, volt_meas)

    A_batch = k * (JL_batch + LJt_batch)
    quad = torch.einsum("ni,nij,nj->n", i, A_batch, i).unsqueeze(1)
    b_pred = k * torch.einsum("jl,nl->nj", J, flux_pred)
    t_pred = quad + 2.0 * (b_pred * i).sum(dim=1, keepdim=True)
    loss_t = nn.functional.mse_loss(t_pred, torq_meas)

    return VOLT_WEIGHT * loss_v / var_v + loss_t / var_t, loss_v, loss_t

## Train

Real co-energy magnitude is small; default init produces O(1) outputs, and
the voltage loss term scales those by `omega` (up to ~1800), which makes
early training numerically unstable unscaled -- the output layer is shrunk
at init, same as the previous version of this notebook did.

In [ ]:
def to_t(a):
    return torch.from_numpy(np.asarray(a, dtype=np.float32)).to(DEVICE)


scaler_X = StandardScaler()
X_train_norm = to_t(scaler_X.fit_transform(X_train))
X_val_norm = to_t(scaler_X.transform(X_val))
X_test_norm = to_t(scaler_X.transform(X_test))
SCALER_SCALE = scaler_X.scale_

i_train_t, i_val_t, i_test_t = to_t(X_train[:, 1:]), to_t(X_val[:, 1:]), to_t(X_test[:, 1:])
omega_train_t, omega_val_t, omega_test_t = to_t(X_train[:, :1]), to_t(X_val[:, :1]), to_t(X_test[:, :1])
volt_train_t, volt_val_t, volt_test_t = to_t(volt_train), to_t(volt_val), to_t(volt_test)
torq_train_t, torq_val_t, torq_test_t = to_t(torq_train), to_t(torq_val), to_t(torq_test)

var_v = volt_train_t.var()
var_t = torq_train_t.var()

model = NeuralFluxPredictor(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, scaler_X, DEVICE, ACTIVATION).to(DEVICE)
with torch.no_grad():
    model.fc2.weight.mul_(1e-2)
    model.fc2.bias.mul_(1e-2)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_val_loss = float("inf")
best_weights = copy.deepcopy(model.state_dict())
epochs_no_improve = 0
stopped_epoch = EPOCHS

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    loss, _, _ = physics_loss(model, X_train_norm, i_train_t, omega_train_t, volt_train_t, torq_train_t, var_v, var_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss, val_lv, val_lt = physics_loss(
            model, X_val_norm, i_val_t, omega_val_t, volt_val_t, torq_val_t, var_v, var_t
        )
    val_loss_val = val_loss.item()

    if val_loss_val < best_val_loss - 1e-6:
        best_val_loss = val_loss_val
        best_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            stopped_epoch = epoch + 1
            break

    if epoch % 1000 == 0:
        print(
            f"  epoch {epoch:5d}  train={loss.item():.4f}  val={val_loss_val:.4f}"
            f"  val_rmse_v={val_lv.item()**0.5:.4f}  val_rmse_t={val_lt.item()**0.5:.4f}"
        )

model.load_state_dict(best_weights)
print(f"\nStopped at epoch {stopped_epoch}, val_loss={best_val_loss:.4f}")

model.eval()
with torch.no_grad():
    _, test_lv, test_lt = physics_loss(
        model, X_test_norm, i_test_t, omega_test_t, volt_test_t, torq_test_t, var_v, var_t
    )
print(f"\n[training-formula eval] Test voltage RMSE [V]: {test_lv.item()**0.5:.4f}")
print(f"[training-formula eval] Test torque  RMSE [Nm]: {test_lt.item()**0.5:.4f}")

In [ ]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
np.save(SCALER_SAVE_PATH, {"mean": scaler_X.mean_, "scale": scaler_X.scale_})
print(f"Saved weights to {MODEL_SAVE_PATH}")
print(f"Saved scaler to {SCALER_SAVE_PATH}")

## Sanity checks against the actual library

Cross-checking against `PMSMDrive.torque()`/`voltage_operator()` directly
(not just this notebook's own training-formula eval) is what caught the
`(J@L)^T` vs `L@J^T` bug in the previous version — kept as standard
practice. Also verifies: inductance is exactly symmetric everywhere (by
construction, not just empirically), and checks how the model behaves at
the specific out-of-training-distribution point that produced the spurious
~13 Nm "torque" with the old architecture.

In [ ]:
cpu = torch.device("cpu")
net, loaded_scaler = load_neural_flux_model(
    MODEL_SAVE_PATH, SCALER_SAVE_PATH,
    hidden_size=HIDDEN_SIZE, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, device=cpu,
    activation=ACTIVATION,
)
neural_flux = NeuralFlux(net, loaded_scaler, cpu, params.L_stat, _build_cross_coupling(2), params.flux_pm)
neural_flux_machine = PMSMDrive(params, flux=neural_flux)

v_pred_lib, t_pred_lib = [], []
for row_idx in range(len(X_test)):
    w = float(X_test[row_idx, 0])
    i_row = X_test[row_idx, 1:]
    t_pred_lib.append(neural_flux_machine.torque(w, i_row))
    v_pred_lib.append(neural_flux_machine.voltage_operator(w, i_row) @ i_row + neural_flux_machine.bemf_dq(w, i_row))
v_pred_lib, t_pred_lib = np.array(v_pred_lib), np.array(t_pred_lib)
rmse_v_lib = np.sqrt(np.mean((volt_test - v_pred_lib) ** 2))
rmse_t_lib = np.sqrt(np.mean((torq_test[:, 0] - t_pred_lib) ** 2))
print(f"[library eval, must match training-formula eval above] Test voltage RMSE [V]: {rmse_v_lib:.4f}")
print(f"[library eval, must match training-formula eval above] Test torque  RMSE [Nm]: {rmse_t_lib:.4f}")

print("\n--- Symmetry check (must hold everywhere, by construction) ---")
rng = np.random.default_rng(0)
max_asym = 0.0
for _ in range(200):
    i_rand = rng.uniform(-30, 30, size=4)
    w_rand = rng.uniform(0, 1800)
    L = neural_flux.inductance(w_rand, i_rand)
    max_asym = max(max_asym, np.max(np.abs(L - L.T)))
print(f"max |L - L.T| over 200 random points: {max_asym:.2e} (should be ~0, i.e. machine precision)")

print("\n--- Extrapolation check: the point that gave ~13 Nm with the old architecture ---")
i_bad = np.array([-8.29179607, -25.51952425, -21.70820393, 15.77193336])
t_old_bug = neural_flux_machine.torque(0.0, i_bad)
t_baseline_only = ieee_machine2().torque(0.0, i_bad)
print(f"torque at this point (new co-energy-residual model): {t_old_bug:.4f} Nm")
print(f"torque at this point (analytic ConstantFlux baseline alone): {t_baseline_only:.4f} Nm")
print("(new model should be reasonably close to the baseline here, not a wild multiple of it)")

## Comparison: new NeuralFlux vs. the old ConstantFlux, same held-out test set

In [ ]:
baseline = ieee_machine2()  # ConstantFlux: single joint-least-squares flux_pm, fixed L_stat


def eval_on_test(mach):
    v_pred, t_pred = [], []
    for row_idx in range(len(X_test)):
        w = float(X_test[row_idx, 0])
        i_row = X_test[row_idx, 1:]
        t_pred.append(mach.torque(w, i_row))
        v_pred.append(mach.voltage_operator(w, i_row) @ i_row + mach.bemf_dq(w, i_row))
    v_pred, t_pred = np.array(v_pred), np.array(t_pred)
    rmse_v = np.sqrt(np.mean((volt_test - v_pred) ** 2))
    rmse_t = np.sqrt(np.mean((torq_test[:, 0] - t_pred) ** 2))
    return rmse_v, rmse_t


rmse_v_const, rmse_t_const = eval_on_test(baseline)
rmse_v_neural, rmse_t_neural = eval_on_test(neural_flux_machine)

print(f"n_test = {len(X_test)}\n")
print(f"{'model':<32} {'voltage RMSE [V]':>18} {'torque RMSE [Nm]':>18}")
print(f"{'ConstantFlux (baseline)':<32} {rmse_v_const:>18.4f} {rmse_t_const:>18.4f}")
print(f"{'NeuralFlux (co-energy residual)':<32} {rmse_v_neural:>18.4f} {rmse_t_neural:>18.4f}")